In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split,Dataset
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
import optuna
from PIL import Image


c:\Users\Lenovo\anaconda3\envs\test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Parrot(Dataset):
    def __init__(self, data_dir):
        self.data = []
        
        self.classes = ['Amazon', 'Gray', 'Macaw', 'White']

        class_folders = [
            'amazon green parrot.jpg',
            'gray parrot.jpg',
            'macaw.jpg',
            'white parrot.jpg'
        ]

        data_dir = os.path.join(data_dir, 'Birds dataset.jpg')

        for label, folder in enumerate(class_folders):
            folder_path = os.path.join(data_dir, folder)

            if not os.path.isdir(folder_path):
                print(f"❌ Not a folder: {folder_path}")
                continue

            for img_name in os.listdir(folder_path):
                if img_name.lower().endswith(('.jpg', '.png', '.jpeg', '.webp')):
                    self.data.append(
                        (os.path.join(folder_path, img_name), label)
                    )

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        print("Total images loaded:", len(self.data))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        return image, label


In [3]:
dataset = Parrot(
    data_dir=r"C:\Users\Lenovo\.cache\kagglehub\datasets\muhammadadeelkaggle\birds-dataset\versions\1"
)

num_classes = len(dataset.classes)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


Total images loaded: 203


In [4]:
from torch.utils.data import DataLoader, random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    pin_memory=True
)


In [5]:
class CNNClassifier(nn.Module):
    def __init__(self, in_channels, dropout):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [7]:
model = CNNClassifier(3,0.3)

In [8]:
def objective(trial):
    batch_size = trial.suggest_int("batch_size", 8, 32, step=8)
    dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    epochs = trial.suggest_int("epochs", 10, 30, step=10)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = CNNClassifier(num_classes, dropout).to(device)

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    criterion = nn.CrossEntropyLoss()

    # Training
    model.train()
    for _ in range(epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return correct / total


In [9]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

print("Best Accuracy:", study.best_value)
print("Best Parameters:", study.best_params)


[I 2026-01-05 18:50:42,878] A new study created in memory with name: no-name-7a3b6ec5-1bf5-4f7c-92ef-7d62a2de412b
c:\Users\Lenovo\anaconda3\envs\test\Lib\site-packages\PIL\Image.py:1043: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[I 2026-01-05 18:51:08,280] Trial 0 finished with value: 0.8048780487804879 and parameters: {'batch_size': 32, 'dropout': 0.2, 'lr': 0.0016451251590983884, 'optimizer': 'SGD', 'epochs': 20}. Best is trial 0 with value: 0.8048780487804879.
[I 2026-01-05 18:51:20,236] Trial 1 finished with value: 0.36585365853658536 and parameters: {'batch_size': 24, 'dropout': 0.5, 'lr': 0.00932113626351768, 'optimizer': 'Adam', 'epochs': 10}. Best is trial 0 with value: 0.8048780487804879.
[I 2026-01-05 18:51:42,784] Trial 2 finished with value: 0.8292682926829268 and parameters: {'batch_size': 24, 'dropout': 0.30000000000000004, 'lr': 0.0004870823335628707, 'optimizer': 'SGD', 'epochs': 20}. Best is tri

Best Accuracy: 0.8292682926829268
Best Parameters: {'batch_size': 24, 'dropout': 0.30000000000000004, 'lr': 0.0004870823335628707, 'optimizer': 'SGD', 'epochs': 20}


In [10]:
import torch.optim as optim

In [11]:
epochs = 10
learning_rate = 0.000191975

criterion = nn.CrossEntropyLoss()
model = CNNClassifier(3, 0.3)   # output classes = 3
model = model

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:   # ✅ use train_loader
    

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()      
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}")


Epoch [1/10]  Loss: 1.4082
Epoch [2/10]  Loss: 1.0399
Epoch [3/10]  Loss: 0.5958
Epoch [4/10]  Loss: 0.3761
Epoch [5/10]  Loss: 0.1253
Epoch [6/10]  Loss: 0.0738
Epoch [7/10]  Loss: 0.0681
Epoch [8/10]  Loss: 0.1113
Epoch [9/10]  Loss: 0.1558
Epoch [10/10]  Loss: 0.0277


In [16]:
# Accuracy

model.eval()
total = 0
corrected = 0
for images , labels in test_loader:
    outputs = model(images)
    _,predicted = torch.max(outputs,dim = 1)
    total += labels.size(0)
    corrected += (predicted == labels).sum().item()
accuracy = corrected/total

print(f'Test Accuracy : {accuracy*100:.2f}%')

Test Accuracy : 14.63%


In [13]:
# Prediction 
img_path = r'C:\Users\Lenovo\OneDrive\Desktop\Data Structures and Algorithms\Deep Learning\Pytorch\parrot-4054102_1280.jpg'
img = Image.open(img_path).convert('RGB')

# apply the existing transform pipeline (data_transforms) and add batch dim
transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
img = transform(img).unsqueeze(0)

# ensure a model exists; if not, create a fresh one (won't be trained)

model = CNNClassifier(num_classes, dropout=0.3)

model.eval()
with torch.no_grad():
	outputs = model(img)
	preds = outputs.argmax(dim=1)

classes = getattr(dataset, "classes", [str(i) for i in range(num_classes)])
predicted = classes[preds.item()]
print(f'Predicted class : {predicted}')

Predicted class : Amazon
